# Coupling tutorial

> **WARNING:** the notebook will take 5 or so minutes (on a CPU) to run to completion.

The goal of this notebook is to demonstrate how one can implement couplings in `stix`. [`Coupling`](https://instadeepai.github.io/stix/api_reference/core/coupling.html)s pair together source and target distributions and can lead to a variety of benefits, most prominently sampling speed. By choosing a coupling which minimizes the amount of energy required to move from one distribution to another, we can incur significantly lower sampling cost. More on this later!

## Structure of the notebook

0. Installation
1. Imports
2. Dataloader
3. Generative Model
    1. Modality registry
    2. Network
    3. Instantiate the Generative Model
4. Training infrastructure
5. Couplings
    1. Product-of-marginals coupling
    2. Mini-batch optimal transport coupling
        1. on the `index` modality
        2. on the `coordinates` modality
        3. noise-to-data via `embedded_source_prior`
    3. Coupling at the dataset level: rectified flow
6. Comparing the couplings:
    1. Cost to train
    2. Cost to sample
    3. Discussion of results


## 0. Installation

We recommend running this notebook in a **fresh virtual environment**. 

Copy the notebook into some new directory. Then, from a terminal, in the new directory containing the notebook (`1.training_and_sampling.ipynb`):
```
python -m venv my_env
source my_env/bin/activate
pip install notebook ipykernel

python -m ipykernel install --user --name my_env --display-name "my_env"

jupyter notebook
```

The next cell installs `stix` from PyPI together with the extra plotting and dataset packages this notebook uses.

In [ ]:
%pip install stix-ml scikit-learn matplotlib seaborn

## 1. Imports

In [ ]:
import copy
import logging
import time
from functools import partial

import grain
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import optax
from flax import nnx
from matplotlib.colors import BoundaryNorm
from scipy.optimize import linear_sum_assignment

from stix.core.coupling import Coupling

# Embedders bridge raw data space and the network's embedding space.
from stix.core.embedder import IdentityEmbedder, OneHotDiscreteEmbedder

# A predefined, ready-to-use generative model: the two-sided counterpart of
# the one used in the introductory tutorial. Its network output *is* the
# velocity, and it implements the methods the rest of the library calls.
from stix.core.gen_model import GenerativeModel
from stix.core.gen_model.factory import VelocityTwoSidedGenerativeModel

# Interpolant classes.
from stix.core.interpolant import FlowMatchingTwoSidedInterpolant

# The modality registry: single source of truth for per-modality config.
from stix.core.modality import ModalityRegistry

# DiT building blocks + the thin EncoderBackboneDecoderNetwork wrapper that orchestrates them.
from stix.nn import (
    DiTBackbone,
    DiTDecoder,
    DiTEncoder,
    EncoderBackboneDecoderNetwork,
    NetworkDimsConfig,
    SumContextEncoder,
    TimeNoiseContextEncoder,
)

# Diffrax-based SDE/ODE solver and direction enum (forward / reverse).
from stix.sampling.solver import Solver, SolverConfig
from stix.sampling.solver_manual import ManualSolver, ManualSolverConfig
from stix.sampling.utils import Direction

# Loss machinery, IO/logging, and the step-based training loop.
from stix.training.loss_pipeline import LossPipeline
from stix.training.training_io_handler import LoggingCategory, TrainingIOHandler
from stix.training.training_loggers import log_metrics_to_line
from stix.training.training_loop import TrainingLoop, TrainingLoopConfig
from stix.typing import Batch, RawSourceTargetPair

In [ ]:
stix_logger = logging.getLogger("stix")
stix_logger.setLevel(logging.INFO)

key = jax.random.PRNGKey(0)

## 2. Dataloader

We use the [grain](https://google-grain.readthedocs.io/en/latest/index.html) library for our dataloaders. See the [data loading tutorial](./2.grain_multimodal_dataloading.ipynb) for in-depth advice on how to set up a dataset in `stix`, and how to use grain.

> **Note:** The grain library enables reproducibility and checkpointing of the dataset state to enable resuming training at the same stage.

Our setup is **two-sided**: we transport ring of Gaussians (source) to another ring (target). These rings are both Gaussian mixture models (GMM). We have two data modalities from each GMM:
- `coordinates` — 2D coordinates of the sampled data point in the $(x, y)$ plane.
- `index` — The index corresponding to the mixture component the coordinates were sampled from.

There is a plot below which visualises the two distribution and both of their modalities.

In [ ]:
# Two rings of Gaussians, both laid out on the vertices of a regular hexagon.
# Only the radius changes between them: the SOURCE ring is tight (radius 1) and
# the TARGET ring is wider (radius 2), sharing the same angular positions.
NUM_MODES = 6
_source_hexagon_angles = jnp.linspace(0.0, 2.0 * jnp.pi, NUM_MODES, endpoint=False)
# Rotate target distribution by 2pi/3 radians:
_target_hexagon_angles = _source_hexagon_angles + (2 / 3) * jnp.pi
_source_hexagon = jnp.stack(
    [jnp.cos(_source_hexagon_angles), jnp.sin(_source_hexagon_angles)], axis=1
)
_target_hexagon = jnp.stack(
    [jnp.cos(_target_hexagon_angles), jnp.sin(_target_hexagon_angles)], axis=1
)
SOURCE_RADIUS = 1.0
TARGET_RADIUS = 2.0
GMM_STD = 0.08
SOURCE_CENTERS = (SOURCE_RADIUS * _source_hexagon).astype(jnp.float32)
TARGET_CENTERS = (TARGET_RADIUS * _target_hexagon).astype(jnp.float32)


def sample_coord_and_index_gmm(
    key: jax.Array, centers: jax.Array
) -> tuple[jax.Array, jax.Array]:
    """Draw one GMM sample: pick a corner, scatter Gaussian noise around it.

    Returns the 2D coordinate and its integer corner index. Wrapping into a
    ``Batch`` (and one-hot encoding the index) happens downstream — this is the
    bare per-point primitive that ``sample_gmm_pool`` vmaps over.
    """
    idx_key, noise_key = jax.random.split(key)
    idx = jax.random.randint(idx_key, (), 0, len(centers))
    coordinates = (
        jax.random.normal(noise_key, (2,), dtype=jnp.float32) * GMM_STD + centers[idx]
    )
    return coordinates, idx

To form a datasampler all one requires is a single sampling method that yields [`Batch`](https://instadeepai.github.io/stix/api_reference/typing/index.html#stix.typing.data.Batch) objects. `Batch` follows a specific structure and we must be careful to match this. Let's now write a wrapper around the sampler defined above which achieves this. 

* Each modality must be wrapped in a `RawSourceTargetPair`. 
    * Each pair now carries a concrete `source` (from the source ring) and `target` (from the target ring): our [`Interpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html) is two-sided (data → data).
* We must specify whether each of the modalities are discrete via the `is_discrete` field.
    * This is propagated through the codebase to enable specific [`Embedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html)s, and to catch errors.

In [ ]:
@jax.jit  # one compiled gather per batch — see the pipeline below
def sample_gmm(
    batch_indices: jax.Array, pool: dict[str, tuple[jax.Array, jax.Array]]
) -> Batch:
    """Read the pooled (source, target) GMM samples at ``batch_indices`` and wrap them in a ``Batch``.

    `grain` shuffles and groups the indices for us; each one selects a paired sample
    from the fixed `pool`, gathered in a single indexing operation per array. The
    `pool` (one per seed) is what distinguishes the train and val streams.

    Two-sided: each modality carries a concrete ``source`` (from the source ring)
    and ``target`` (from the target ring). Discrete modalities are one-hot (the
    codebase convention), and ``is_discrete`` on the ``Batch`` tells the registry
    which modalities are categorical.
    """
    coordinates_source, coordinates_target = pool["coordinates"]
    index_source, index_target = pool["index"]
    raw_batch = {
        "coordinates": RawSourceTargetPair(
            source=coordinates_source[batch_indices],
            target=coordinates_target[batch_indices],
        ),
        "index": RawSourceTargetPair(
            source=index_source[batch_indices], target=index_target[batch_indices]
        ),
    }
    return Batch(
        raw_batch=raw_batch,
        is_discrete={"coordinates": False, "index": True},
    )

To compare the different coupling strategies like-for-like we use a fixed number of samples (`DATASET_SIZE`).

Finally, we wire our GMM sampler into a grain dataset. The order below is the one grain recommends, and it is the same in every notebook here:

- **seed** — fix the random key of the dataset for reproducibility;
- **shuffle** — shuffles the data;
- **repeat** — transform the dataset of fixed length into an infinite iterator;
- **batch** — groups indices into a batch; `drop_remainder=True` keeps the shape static, so `jax.jit` compiles the step once;
- **map** — applies `sample_gmm` to the batched indices, gathering one `Batch` out of the pool;
- **to_iter_dataset** — hands back the Python iterator we pull batches from.

Two things here are worth copying into your own pipelines. First, **grain chooses and groups elements; JAX makes the arrays.** grain itself runs on the CPU, so we give it only indices to shuffle, and a single jitted call gathers a batch of rows out of the pool. Indexing the pool one sample at a time instead means one round-trip to the accelerator per sample, which for a toy dataset like this one costs more than the training step it feeds.

Second, `to_iter_dataset` goes **last**: everything above it supports random access, which is what lets `batch` group by slicing rather than by pulling elements through an iterator one at a time.


In [ ]:
DATASET_SIZE = 50_000
VALIDATION_DATASET_SIZE = DATASET_SIZE // 20
# Small batches on purpose: at this size a training step is cheap, so the same
# wall clock buys far more optimiser steps — which is what these models need.
batch_size = 64


def sample_gmm_pool(size: int, seed: int) -> dict[str, tuple[jax.Array, jax.Array]]:
    """Draw a fixed pool of ``size`` paired (source, target) GMM samples once, deterministic in ``seed``.

    Returns, per modality, a ``(source, target)`` pair of arrays — the concrete
    two-sided dataset `grain` then shuffles, repeats and batches. Source and
    target are drawn independently here; the coupling (product-of-marginals vs
    MBOT) decides how they are paired at loss time.
    """
    source_key, target_key = jax.random.split(jax.random.key(seed))
    source_keys = jax.random.split(source_key, size)
    target_keys = jax.random.split(target_key, size)
    coordinates_source, index_source = jax.vmap(
        lambda k: sample_coord_and_index_gmm(k, SOURCE_CENTERS)
    )(source_keys)
    coordinates_target, index_target = jax.vmap(
        lambda k: sample_coord_and_index_gmm(k, TARGET_CENTERS)
    )(target_keys)
    num_modes = len(TARGET_CENTERS)
    # The pool lives on the device, as one array per modality and side. `grain`
    # only ever shuffles *indices* into it, and `sample_gmm` gathers a whole batch
    # of rows in one go — so the pool is drawn once and never leaves the accelerator.
    return {
        "coordinates": (coordinates_source, coordinates_target),
        "index": (
            jax.nn.one_hot(index_source, num_modes),
            jax.nn.one_hot(index_target, num_modes),
        ),
    }


def gmm_dataset(size: int, seed: int):
    """Helper function to instantiate a grain dataset over a fixed sample pool."""
    pool = sample_gmm_pool(size, seed=seed)
    return (
        grain.MapDataset.range(size)
        .seed(seed)
        .shuffle()
        .repeat()
        .batch(batch_size, drop_remainder=True)
        .map(partial(sample_gmm, pool=pool))
        .to_iter_dataset()
    )


train_iter = iter(gmm_dataset(DATASET_SIZE, seed=0))
validation_iter = iter(gmm_dataset(VALIDATION_DATASET_SIZE, seed=42))

Before going further, let's visualise what we're transporting. Below are the **source** ring $\rho_0$ (six modes on a hexagon, radius 1) and the **target** ring $\rho_1$ (same hexagonal layout, radius 2), each point coloured by its GMM component `index`. The shared colourbar lets you read off which mode is which. The coupling is precisely what decides how to pair source samples with target samples.

In [ ]:
# Draw a small pool purely for visualisation and colour each point by its GMM
# component index (the discrete `index` modality the model also learns).
viz_pool = sample_gmm_pool(size=2000, seed=0)
source_coords, target_coords = viz_pool["coordinates"]
source_index = jnp.argmax(viz_pool["index"][0], axis=-1)
target_index = jnp.argmax(viz_pool["index"][1], axis=-1)

cmap = plt.get_cmap("tab10", NUM_MODES)
norm = BoundaryNorm([i - 0.5 for i in range(NUM_MODES + 1)], NUM_MODES)

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
for ax, coords, index, title in [
    (axes[0], source_coords, source_index, r"Source $\rho_0$"),
    (axes[1], target_coords, target_index, r"Target $\rho_1$"),
]:
    scatter = ax.scatter(
        coords[:, 0], coords[:, 1], s=8, c=index, cmap=cmap, norm=norm, alpha=0.6
    )
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")

fig.colorbar(scatter, ax=axes, label="mode index", ticks=range(NUM_MODES))
plt.show()

## 3. Generative Model

The [`GenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html) ties together the interpolant, embedders, network, and loss, and exposes crucial training and sampling methods that everything else calls.

Constructing one requires a [`ModalityRegistry`](https://instadeepai.github.io/stix/api_reference/core/modality.html) and a [`Network`](https://instadeepai.github.io/stix/api_reference/nn/network.html#stix.nn.Network). For precise details on how to set up a `Generative Model` refer to the [introductory tutorial](./1.training_and_sampling.ipynb).


### 3.1. Modality registry

In [ ]:
batch = next(train_iter)
# Build the registry straight from a batch: `from_batch` reads each modality's
# shape from the (one-hot) data and its discreteness from `batch.is_discrete`.
# The remaining fields (interpolant, embedder, loss_fn) are filled in the
# sections below.
modality_registry = ModalityRegistry.from_batch(batch)

#### Interpolant, Embedder, Loss

1. **[`Interpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html)**: flow matching interpolant ([Lipman et. al. 2022](https://openreview.net/forum?id=PqvMRDCJT9t)).
2. **[`Embedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html)**: we use the following embedders
    - [`IdentityEmbedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html#stix.core.embedder.IdentityEmbedder): no-op, for already-continuous modalities.
    - [`OneHotDiscreteEmbedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html#stix.core.embedder.OneHotDiscreteEmbedder): treats one-hot vectors as continuous embeddings.
3. **Loss**: We train the network to predict the **velocity**, and thus we create a loss which is simple the MSE between the network output and the interpolant's *conditional velocity*. 

We use the `.set` method to perform these replacements.

The interpolant we use is the [`FlowMatchingTwoSidedInterpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html#stix.core.interpolant.FlowMatchingTwoSidedInterpolant) which has $z_t = (1-t)z_{\mathrm{src}} + t z_{\mathrm{tgt}}$ (no noise term!).

In [ ]:
# Two-sided deterministic Flow Matching: z_t = (1 - t) z_src + t z_tgt (no noise term).
interpolant = FlowMatchingTwoSidedInterpolant()

# Broadcast the (shared) interpolant onto every modality in the registry.
modality_registry.set("interpolant", interpolant)

In [ ]:
# Embedders are shape-dependent, so install them with per-modality factories.
# Two `set` calls partition the modalities on `is_discrete` (exhaustive for a
# bool flag): discrete -> OneHotDiscreteEmbedder, continuous -> IdentityEmbedder.
modality_registry.set(
    field="embedder",
    value=lambda modality: OneHotDiscreteEmbedder(dm_shape=modality.shape),
    is_factory=True,
    filter_fn=lambda modality: modality.is_discrete,
    use_deepcopy=True,
)
modality_registry.set(
    field="embedder",
    value=lambda modality: IdentityEmbedder(dm_shape=modality.shape),
    is_factory=True,
    filter_fn=lambda modality: not modality.is_discrete,
    use_deepcopy=True,
)

# Generator type: the two-sided interpolant is deterministic (no score), so each
# modality yields a plain `Velocity` generator, enough for ODE sampling.

### 3.2. Network

We will use a [`EncoderBackboneDecoderNetwork`](https://instadeepai.github.io/stix/api_reference/nn/network.html#stix.nn.EncoderBackboneDecoderNetwork), but we stress that the user is **free to substitute** the network with some other `Network` subclass (any `nnx.Module` that implements the `Network` contract).

In [ ]:
key, model_key = jr.split(key)
rngs = nnx.Rngs(model_key)

# Some dimensions shared across the encoder, backbone, decoder and context encoder.
# Declared once here, then injected into each component of the network.
network_dims = NetworkDimsConfig(
    embedding_dim=128,
    context_dim=64,
    ffn_hidden_dim=256,
)

In [ ]:
# Per-modality encoders/decoders + shared backbone, all injected with network_dims.

encoders = modality_registry.map(
    lambda modality: DiTEncoder(
        network_dims, input_dim=modality.embedder.embedding_shape[-1], rngs=rngs
    )
)
decoders = modality_registry.map(
    lambda modality: DiTDecoder(
        network_dims, output_dim=modality.embedder.embedding_shape[-1], rngs=rngs
    )
)
backbone = DiTBackbone(
    network_dims,
    modality_num_tokens=modality_registry.broadcast(1),
    rngs=rngs,
)

# Context encoder. The interpolant is deterministic, so there is no noise
# schedule to hand over, but TimeNoiseContextEncoder still needs a
# strictly-positive `gamma_fn` for its log-gamma feature.
# We pass `t` itself. Yes: this carries no information the direct time feature
# does not already provide.
context_encoder = SumContextEncoder(
    time_encoder=TimeNoiseContextEncoder(
        network_dims,
        gamma_fn=lambda t: t,
        rngs=rngs,
    )
)

In [ ]:
network = EncoderBackboneDecoderNetwork(
    encoders=encoders,
    backbone=backbone,
    decoders=decoders,
    context_encoder=context_encoder,
)

### 3.3. Instantiate the Generative Model

With the `ModalityRegistry` fully populated and the network built, we instantiate the `GenerativeModel` that ties them together.

We use the predefined [`VelocityTwoSidedGenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html), the two-sided counterpart of the `VelocityOneSidedGenerativeModel` which was used in the [introductory tutorial](./1.training_and_sampling.ipynb). As we note in the [generative modelling tutorial](./3.generative_model.ipynb), the loss completely determines the network output, and from this it is the user's responsiblity to specify how `get_generator` converts it into the sample-time generator.

Because its loss trains the network on velocity, and because the two-sided interpolant is deterministic (the score is undefined), `get_generator` returns a plain [`Velocity`](https://instadeepai.github.io/stix/api_reference/core/generator.html#stix.core.generator.Velocity) generator — the velocity is simply the identity of the network output. The deterministic two-sided interpolant infers `generator_type=Velocity`.

> ***Recall: a `Velocity` generator carries no score, so it only supports deterministic (ODE) sampling. SDE sampling requires a `VelocityAndScore` generator (and an interpolant for which the score is defined).***

In [ ]:
gen_model = VelocityTwoSidedGenerativeModel(
    network=network,
    modality_registry=modality_registry,
)

## 4. Training infrastructure

With most of our main objects instantiated, let's define the remaining ones required to run the training:

- **Optimizer**: Optax optimizer.

- [**IO Handler**](https://instadeepai.github.io/stix/api_reference/training/io_handling.html#stix.training.training_io_handler.TrainingIOHandler): The IO handler that takes care of the logging and checkpointing.

- [**Training Loop**](https://instadeepai.github.io/stix/api_reference/training/training_loop.html#stix.training.training_loop.TrainingLoop): It pulls batches from the data iterators for `num_steps`, keeps an EMA copy of the parameters (used for evaluation and sampling).

In [ ]:
# How long we train, and how often we stop to evaluate.
NUM_STEPS = 3000
EVAL_EVERY_N_STEPS = 100

# The optimiser.
# We use a decaying schedule as this improves results.
LEARNING_RATE = 3e-3
learning_rate_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=LEARNING_RATE,
    warmup_steps=NUM_STEPS // 20,
    decay_steps=NUM_STEPS,
)
optimizer_tx = optax.adam(learning_rate_schedule)

In [ ]:
def make_history_io_handler(
    verbose: bool = True,
) -> tuple[list[dict], TrainingIOHandler]:
    """Build a TrainingIOHandler that records a plottable training history.

    Returns a (history, io_handler) pair: pass io_handler to TrainingLoop, and
    history fills in place with {"step", "train_loss", "eval_loss"} records as
    training runs. With verbose=True, metrics are also printed to the console.
    """
    history: list[dict] = []
    by_step: dict[int, dict] = {}

    def _collect(category: LoggingCategory, to_log: dict, step: int) -> None:
        entry = by_step.get(step)
        if entry is None:
            entry = {"step": step}
            by_step[step] = entry
            history.append(entry)
        if category == LoggingCategory.TRAIN_METRICS:
            entry["train_loss"] = to_log["loss"]
        elif category == LoggingCategory.EVAL_METRICS:
            entry["eval_loss"] = to_log["loss"]

    io_handler = TrainingIOHandler()
    io_handler.attach_logger(_collect)
    if verbose:
        io_handler.attach_logger(log_metrics_to_line)
    return history, io_handler

In [ ]:
training_loop_cfg = TrainingLoopConfig(
    num_steps=NUM_STEPS,
    eval_every_n_steps=EVAL_EVERY_N_STEPS,
)

# Wall-clock cost of each coupling, filled in as we train. Every coupling runs
# the same number of steps, so differences here are the cost of the coupling
# itself — MBOT solves an assignment problem on every batch, and rectified flow
# has to train a first model before it can start.
training_seconds: dict[str, float] = {}


def run_and_time(training_loop: TrainingLoop, name: str) -> None:
    """Run a training loop, recording its wall-clock time under ``name``."""
    start = time.perf_counter()
    training_loop.run()
    training_seconds[name] = time.perf_counter() - start

The only thing that remains for training is the [`LossPipeline`](https://instadeepai.github.io/stix/api_reference/training/loss_pipeline.html) which computes the per-batch training loss, and crucially the `Coupling`. For a detailed reminder of how this works, please see the [introductory tutorial](./1.training_and_sampling.ipynb).

## 5. Couplings

`Coupling`s pair together the source $p_{\mathrm{src}}(z_{\mathrm{src}})$ and target $p_{\mathrm{tgt}}(z_{\mathrm{tgt}})$ distributions via $\pi(z_{\mathrm{src}},z_{\mathrm{tgt}})$. The loss is computed on samples $z_t$ drawn from $p_{t}(z_t|z_{\mathrm{src}},z_{\mathrm{tgt}})$, which is defined by the specific interpolant - in our codebase the method [`Interpolant.interpolate`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html#stix.core.interpolant.Interpolant.interpolate), and these $(z_{\mathrm{src}},z_{\mathrm{tgt}}) \sim \pi(z_{\mathrm{src}},z_{\mathrm{tgt}})$.

Thus, the marginal distribution of $z_t$ is $p_t(z_t) = \int p_{t}(z_t|z_{\mathrm{src}},z_{\mathrm{tgt}})\pi(z_{\mathrm{src}},z_{\mathrm{tgt}}) \mathrm{d}z_{\mathrm{src}}\mathrm{d}z_{\mathrm{tgt}}$.

> **Why couplings must not touch interpolant noise $\varepsilon$:**
> For a continuous stochastic interpolant $z_t = I_t(z_{\mathrm{src}}, z_{\mathrm{tgt}}) + \gamma_t\,\varepsilon$, the conditional score is $s_t = -\varepsilon / \gamma_t$ **only if** $\varepsilon$ is independent of $(z_{\mathrm{src}}, z_{\mathrm{tgt}})$. Correlating $\varepsilon$ to $(z_{\mathrm{src}}, z_{\mathrm{tgt}})$ would thus **invalidate SDE sampling** (ODE sampling would still look fine). Couplings in `stix` therefore re-pair **source and target only**; $\varepsilon$ is drawn *after* coupling. Consequently, for **one-sided interpolants** couplings cannot be used to correlate the initial and final variables, as the initial variables $z_{t=0}$ are realized from $\varepsilon$ (e.g. $z_t = \beta_t z_{\mathrm{tgt}} + \gamma_t\,\varepsilon$). Instead, one can use a **two-sided** interpolant and draw $z_{\mathrm{src}}$ from an `embedded_source_prior` (§5.2.3).

In practice, at training-time a data loader yields batches of source-target pairs. To simulate a sampling from a joint distribution $\pi$, the `Coupling` object provides a mean to re-pair the data, introducing correlations between the source and target variables of each batch.

> **Online vs. offline coupling:**
> The data loader can be built such that the pairs are *already correlated*, simulating a sampling from a specific joint distribution. We refer to this kind of coupling as an **offline** coupling. The `Coupling` object can only be used for **online** coupling, namely couplings introducing correlations at the level of each batch.

Unless otherwise stated, in this tutorial we use a data loader producing pairs with independent source and target variables.

We will introduce three different couplings to display the different behaviour:

1. Product-of-marginals coupling (**the default**, `coupling=None`): The most simple coupling of all, $\pi(z_{\mathrm{src}},z_{\mathrm{tgt}}) = p_{\mathrm{src}}(z_{\mathrm{src}})p_{\mathrm{tgt}}(z_{\mathrm{tgt}})$. No `Coupling` is applied; source and target stay independently paired.
2. Mini-batch optimal transport (MBOT) coupling (`stix` ships no MBOT coupling of its own, because the transport cost is a modelling choice; we write one below): The coupling is computed on-the-fly on batches during the computation of the loss in [`LossPipeline`](https://instadeepai.github.io/stix/api_reference/training/loss_pipeline.html). We write two data→data implementations, plus a noise→data sketch with `embedded_source_prior`:
    1. Pair based on distance of the 'index' modality.
    2. Pair based on the distance of the 'coordinates' modality.
    3. Noise→data via a two-sided interpolant and `embedded_source_prior` (§5.2.3).
3. Couple at the dataset level: Another option is to pair points at the dataset level, where each source and target are paired. As we will go on to see, we could use a optimal transport criteria, but instead we use the popular Rectified Flow, as defined in [Liu et al. 2022](https://arxiv.org/abs/2209.03003): The coupling is obtained from a trained model. In particular, we train a model with no coupling (independent pairing); this learns a map from data to noise via the deterministic ODE sampler. We use this to determine the coupling, in our code this means switching to a two-sided model **with no noise**, where the source (noise) and target are coupled at the dataset level, then re-training.

That is three couplings but **four models**, since the MBOT coupling gets two implementations. We will demonstrate how to run all four, and then compare their sampling performance.

### 5.1. Product-of-marginals coupling

The [`LossPipeline`](https://instadeepai.github.io/stix/api_reference/training/loss_pipeline.html) computes the per-batch training loss, and it deals with any `Coupling`. For a detailed reminder of how this works, please see the [introductory tutorial](./1.training_and_sampling.ipynb).

We will use the default (`coupling=None`: independent product-of-marginals pairing).

In [ ]:
loss_pipeline_POM = LossPipeline()

In [ ]:
gen_model_POM = copy.deepcopy(gen_model)

logs_history_POM, io_handler_POM = make_history_io_handler()

training_loop_POM = TrainingLoop(
    train_data=train_iter,
    val_data=validation_iter,
    loss_pipeline=loss_pipeline_POM,
    gen_model=gen_model_POM,
    optimizer_tx=optimizer_tx,
    config=training_loop_cfg,
    io_handler=io_handler_POM,
)
run_and_time(training_loop_POM, "Product-of-marginals")

### 5.2. MBOT coupling

Under the product-of-marginals coupling above, each source point and target points are paired independently, i.e. $\pi(z_{\mathrm{src}},z_{\mathrm{tgt}}) = p_{\mathrm{src}}(z_{\mathrm{src}})p_{\mathrm{tgt}}(z_{\mathrm{tgt}})$. **Mini-batch optimal transport (MBOT)** does something smarter: within every mini-batch it looks at *all* the source and target points at once and re-pairs them so that paired points are as **close as possible**, according so some criteria we specify.

Why bother? The model learns a velocity field transporting the source distribution to the target distribution. In a sense, this field locally averages the transport path associated to each pair. If pairs are close, the trajectories following the velocity field will be short and straight, so the solver will need **fewer steps** to sample accurately. Random pairing typically produces long and crossing paths that are expensive to integrate.

Concretely, given a batch of source points $\{z_{\mathrm{src}}^i\}_{i}$ and target points $\{z_{\mathrm{tgt}}^j\}_{j}$, together with a **cost** $c(z_{\mathrm{src}}, z_{\mathrm{tgt}})$ that we are free to choose, MBOT finds the one-to-one pairing, a permutation $\sigma$, that minimises the total cost $\sum_i c(z_{\mathrm{src}}^i, z_{\mathrm{tgt}}^{\sigma(i)})$. This is the classic **optimal-transport assignment problem**, solved fresh for each batch inside the loss. *Any* cost can be used; a common choice, and the one we use here, is the squared distance $c(z_{\mathrm{src}}, z_{\mathrm{tgt}}) = || z_{\mathrm{src}} - z_{\mathrm{tgt}} ||^2$.

Two design choices then remain: the cost $c$, and what to apply it to. Here we keep $c$ as the squared distance, and compute $c$ over different modalities: 1. pairing by the `index` modality (§5.2.1), and 2. by the `coordinates` modality (§5.2.2).

> ***We note here that we could perform this optimal transport pairing at the dataset level, rather than at the minibatch level. This is investigated in §5.3***

#### Implementing the coupling

The recipe is short:

1. **Cost matrix.** Compute $C_{ij} = c(x_0^i, x_1^j)$ for every source $i$ and target $j$ on the chosen modality — a `(batch, batch)` matrix. Any cost $c$ works; here we take it to be the squared distance $\lVert x_0^i - x_1^j \rVert^2$.
2. **Solve the assignment.** Find the cheapest one-to-one pairing. We use SciPy's [`linear_sum_assignment`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linear_sum_assignment.html) (the Hungarian algorithm), which solves this problem *exactly*.
3. **Reorder.** Apply the resulting permutation to every modality's target; the source stays put. The pairs are now $(x_0^i, x_1^{\sigma(i)})$. Interpolant noise $\varepsilon$ is **not** part of the coupling — the loss pipeline samples it independently afterwards.

Two small JAX details:

- We wrap the call to `scipy.optimize.linear_sum_assignment` in a [`jax.pure_callback`](https://docs.jax.dev/en/latest/external-callbacks.html), this solves two problems:
    1. The loss runs inside `jax.jit`, but `linear_sum_assignment` is a plain SciPy/NumPy function. `jax.pure_callback` allows the function to be called.
    2. The operations in `linear_sum_assignment` require concrete values: Its algorithm is data-dependent (the steps it takes depend on the actual cost values), so it needs real numbers, not the abstract tracers `jax.jit` works with, and `jax.pure_callback` supplies these concrete values at runtime.
- The pairing is a *discrete* (non-differentiable) decision, so no gradient should flow through it — we wrap the cost matrix in `jax.lax.stop_gradient`.

We subclass [`Coupling`](https://instadeepai.github.io/stix/api_reference/core/coupling.html#stix.core.coupling.Coupling) — the library's abstract base class — with **one** class parameterised by `cost_modality`, then instantiate it twice in the two subsections below.

In [ ]:
def _solve_assignment_problem(cost_matrix: np.ndarray) -> np.ndarray:
    """Return the target index matched to each source, minimising total cost.

    Solves the balanced linear assignment problem exactly (the Hungarian
    algorithm). The result is a permutation of ``range(batch_size)``.
    """
    _source_indices, target_indices = linear_sum_assignment(cost_matrix)
    return target_indices.astype(np.int32)


def optimal_assignment(cost_matrix: jax.Array) -> jax.Array:
    """Exact optimal-transport permutation for a ``(batch, batch)`` cost matrix.

    ``linear_sum_assignment`` is a SciPy (NumPy) routine while the loss runs
    inside ``jax.jit``; ``jax.pure_callback`` lets the jitted computation call
    back out to ordinary host Python. See the JAX external-callbacks guide:
    https://docs.jax.dev/en/latest/external-callbacks.html
    """
    batch_size = cost_matrix.shape[0]
    # The pairing is a discrete decision, no gradient should flow through it.
    cost_matrix = jax.lax.stop_gradient(cost_matrix)
    return jax.pure_callback(
        _solve_assignment_problem,
        jax.ShapeDtypeStruct((batch_size,), jnp.int32),
        cost_matrix,
    )


class ModalityMBOTCoupling(Coupling):
    """Mini-batch optimal-transport coupling (two-sided).

    Re-pairs source and target samples *within each batch* so that paired
    points are as close as possible under one modality's geometry, replacing
    the random pairing of the product-of-marginals coupling. Closer pairs give
    straighter transport paths, which need fewer solver steps to sample.

    The pairing is driven by a single ``cost_modality`` (``"index"`` or
    ``"coordinates"`` here): distances are measured on that modality alone, and
    the resulting permutation is applied to *every* modality so the batch stays
    aligned.
    """

    def __init__(self, cost_modality: str):
        """Store which modality's distance drives the transport cost."""
        self.cost_modality = cost_modality

    def __call__(self, raw_batch, embedded_pairs):
        """Reorder targets by the optimal within-batch assignment."""
        # Cost from the chosen modality's raw samples. ``raw_batch`` is the
        # pre-embedding input data, independent of the network parameters, so
        # differentiating the loss never reaches the assignment.
        source = raw_batch[self.cost_modality].source
        target = raw_batch[self.cost_modality].target

        # Squared distance between every (source i, target j) pair -> (B, B),
        # treating each sample as a flat feature vector.
        cost_matrix = ((source[:, None, :] - target[None, :, :]) ** 2).sum(-1)

        # One permutation of the target batch, shared across all modalities.
        permutation = optimal_assignment(cost_matrix)

        # Keep source fixed; reorder each modality's target (raw + embedded).
        # Interpolant noise is sampled after coupling and is never permuted.
        reordered_raw = {
            name: pair._replace(target=pair.target[permutation])
            for name, pair in raw_batch.items()
        }
        reordered_embedded = {
            name: pair._replace(target=pair.target[permutation])
            for name, pair in embedded_pairs.items()
        }
        return reordered_raw, reordered_embedded

#### 5.2.1. Couple based on index modality

Here the cost is the distance between the one-hot **`index`** vectors: it is $0$ when two points share the same mixture component and positive otherwise. The assignment therefore pairs each source point with a target point from the **same mode**.

Because the target ring is rotated by $2\pi/3$, same-mode points sit far apart — so this coupling is *semantically* aligned (mode $k \to$ mode $k$) but the transport it induces is geometrically long.

In [ ]:
loss_pipeline_MBOT_index = LossPipeline(
    coupling=ModalityMBOTCoupling(cost_modality="index")
)

In [ ]:
gen_model_MBOT_index = copy.deepcopy(gen_model)

logs_history_MBOT_index, io_handler_MBOT_index = make_history_io_handler()

training_loop_MBOT_index = TrainingLoop(
    train_data=train_iter,
    val_data=validation_iter,
    loss_pipeline=loss_pipeline_MBOT_index,
    gen_model=gen_model_MBOT_index,
    optimizer_tx=optimizer_tx,
    config=training_loop_cfg,
    io_handler=io_handler_MBOT_index,
)
run_and_time(training_loop_MBOT_index, "MBOT (index)")

#### 5.2.2. Couple based on coordinates modality

Here the cost is the squared Euclidean distance between the 2D **`coordinates`**. The assignment pairs each source point with the **spatially nearest** target point — which, thanks to the $2\pi/3$ rotation, generally belongs to a *different* mode.

This yields the shortest total transport (the straightest paths, cheapest to sample), at the price of scrambling the mode identities. We will see the payoff when we compare sampling below.

In [ ]:
loss_pipeline_MBOT_coordinates = LossPipeline(
    coupling=ModalityMBOTCoupling(cost_modality="coordinates")
)

In [ ]:
gen_model_MBOT_coordinates = copy.deepcopy(gen_model)

logs_history_MBOT_coordinates, io_handler_MBOT_coordinates = make_history_io_handler()

training_loop_MBOT_coordinates = TrainingLoop(
    train_data=train_iter,
    val_data=validation_iter,
    loss_pipeline=loss_pipeline_MBOT_coordinates,
    gen_model=gen_model_MBOT_coordinates,
    optimizer_tx=optimizer_tx,
    config=training_loop_cfg,
    io_handler=io_handler_MBOT_coordinates,
)
run_and_time(training_loop_MBOT_coordinates, "MBOT (coordinates)")

#### 5.2.3. Noise-to-data via `embedded_source_prior`

The main example in this notebook is **data→data**: both rings are real distributions with a raw `source` in every batch. What if you want **noise→data** (Gaussian source) *and* a non-trivial coupling?

You **cannot** keep a one-sided interpolant and MBOT-correlate $\varepsilon$ with $z_{\mathrm{tgt}}$ — that breaks $s_t = -\varepsilon/\gamma_t$ (see the note in §5). Instead:

1. Use a **two-sided** interpolant (here still `FlowMatchingTwoSidedInterpolant`).
2. Leave `raw.source = None` in the dataloader.
3. Set each modality's `embedded_source_prior` to a callable `(key, shape) -> array` (e.g. `jr.normal`). The loss pipeline fills $z_{\mathrm{src}}$ in **embedding space** before coupling; at sampling time the same prior is used by [`ModalityRegistry.sample_initial_state`](https://instadeepai.github.io/stix/api_reference/core/modality.html).

This approach allows you to choose the law of $z_{\mathrm{src}}$ directly in embedding space, rather than inventing a raw $x_{\mathrm{src}}$ and pushing it through the embedder.

When the source comes from a prior, **MBOT cost must use embedded pairs** (`raw.source` is `None`). The small coupling below does exactly that.

In [ ]:
# Noise→data demo: same target ring, Gaussian z_src from embedded_source_prior.
# We keep only the continuous ``coordinates`` modality for brevity.


class EmbeddedMBOTCoupling(Coupling):
    """MBOT that costs on *embedded* pairs (required when raw.source is None)."""

    def __init__(self, cost_modality: str):
        """Store which modality's embedded distance drives the transport cost."""
        self.cost_modality = cost_modality

    def __call__(self, raw_batch, embedded_pairs):
        """Reorder targets by the optimal within-batch assignment on embeddings."""
        source = embedded_pairs[self.cost_modality].source
        target = embedded_pairs[self.cost_modality].target
        cost_matrix = ((source[:, None, :] - target[None, :, :]) ** 2).sum(-1)
        permutation = optimal_assignment(cost_matrix)
        reordered_raw = {
            name: pair._replace(target=pair.target[permutation])
            for name, pair in raw_batch.items()
        }
        reordered_embedded = {
            name: pair._replace(target=pair.target[permutation])
            for name, pair in embedded_pairs.items()
        }
        return reordered_raw, reordered_embedded


def _noise_to_data_batch(pool, batch_indices: np.ndarray) -> Batch:
    """Target-only batch: source is drawn later from embedded_source_prior."""
    _coordinates_source, coordinates_target = pool["coordinates"]
    return Batch(
        raw_batch={
            "coordinates": RawSourceTargetPair(
                source=None, target=coordinates_target[batch_indices]
            ),
        },
        is_discrete={"coordinates": False},
    )


# Local pool for this demo (grain keeps its own pool inside ``gmm_dataset``).
noise_to_data_pool = sample_gmm_pool(batch_size, seed=1)
demo_indices = np.arange(batch_size)

# Registry: two-sided FM + Gaussian prior in embedding space.
noise_to_data_registry = ModalityRegistry.from_batch(
    _noise_to_data_batch(noise_to_data_pool, demo_indices)
)
noise_to_data_registry.set("interpolant", FlowMatchingTwoSidedInterpolant())
noise_to_data_registry.set(
    "embedder",
    lambda m: IdentityEmbedder(dm_shape=m.shape),
    is_factory=True,
)
noise_to_data_registry.set("embedded_source_prior", jr.normal)

# Reuse the same NetworkDimsConfig; fresh RNG stream for this demo network.
key, ntd_key = jr.split(key)
ntd_rngs = nnx.Rngs(ntd_key)
ntd_encoders = noise_to_data_registry.map(
    lambda modality: DiTEncoder(
        network_dims, input_dim=modality.embedder.embedding_shape[-1], rngs=ntd_rngs
    )
)
ntd_decoders = noise_to_data_registry.map(
    lambda modality: DiTDecoder(
        network_dims, output_dim=modality.embedder.embedding_shape[-1], rngs=ntd_rngs
    )
)
ntd_backbone = DiTBackbone(
    network_dims,
    modality_num_tokens=noise_to_data_registry.broadcast(1),
    rngs=ntd_rngs,
)
ntd_context = SumContextEncoder(
    time_encoder=TimeNoiseContextEncoder(
        network_dims,
        gamma_fn=lambda t: t,
        rngs=ntd_rngs,
    )
)
noise_to_data_network = EncoderBackboneDecoderNetwork(
    encoders=ntd_encoders,
    backbone=ntd_backbone,
    decoders=ntd_decoders,
    context_encoder=ntd_context,
)

noise_to_data_model = VelocityTwoSidedGenerativeModel(
    network=noise_to_data_network,
    modality_registry=noise_to_data_registry,
)
noise_to_data_pipeline = LossPipeline(
    coupling=EmbeddedMBOTCoupling(cost_modality="coordinates")
)

# One training step: prior fills z_src, MBOT re-pairs (z_src, z_tgt), then loss.
demo_batch = _noise_to_data_batch(noise_to_data_pool, demo_indices)
key, loss_key = jr.split(key)
demo_loss, _ = noise_to_data_pipeline(noise_to_data_model, demo_batch, loss_key)
print(f"noise→data + embedded MBOT: loss = {float(demo_loss):.4f}")

# Sampling: same helper as one-sided models, but z_src} comes from the prior.
key, init_key = jr.split(key)
z0_from_prior = noise_to_data_registry.sample_initial_state(init_key, num_samples=4)
print("sample_initial_state shapes:", {k: v.shape for k, v in z0_from_prior.items()})

### 5.3. Couple at the dataset level: rectified flow

See [Liu et al. (2023)](https://openreview.net/forum?id=XVjTT1nw5z) for a comprehensive overview of the rectified flow framework. In short:
1. Train a standard model (e.g. the product-of-marginals above).
2. Sample from the model using the ODE. As this is deterministic mapping, by definition we have that the trajectories $(z_t,t)$ are non overlapping.
3. Sample either forward or backward in time and save generated points, $\{(x_0^{i}, x_1^{i})\}_{i=1}^{N}$.
4. Train a model on this new dataset. The dataset, $\{(x_0^{i}, x_1^{i})\}_{i=1}^{N}$, in essence, defines a coupling but at the *dataset level*.

> In this notebook we use a very simple and naive implementation of Rectified flow, please see [Liu et al. (2023)](https://openreview.net/forum?id=XVjTT1nw5z) and subsequent papers for state-of-the-art formulations of this methodology.

To make the comparisons later fair, let's sample exactly $N=$`DATASET_SIZE` samples to form the new dataset. Below is some standard code to sample unconditionally from the model.

> ***Note: The `from_embeddings_to_raw` operation used to recover $x$ from $z$ for discrete modalities is not bijective and thus we don't fulfil the assumptions of Rectified flow. This being said, we proceed anyway as this is for demonstrative purposes.***

The [**`Solver`**](https://instadeepai.github.io/stix/api_reference/sampling/solver.html#stix.sampling.solver.Solver) integrates the learned drift along $t \in [0, 1]$ with [`diffrax`](https://docs.kidger.site/diffrax/). To run the solver, the parameters you need to define are the following : 

- **`num_samples`** : The number of samples you want to sample at once, i.e. the batch size of the solver. You could run it multiple times with a different seed to get more samples.

- **`max_solver_steps`** : The maximum number of steps allowed for the `diffrax` solver. This corresponds to a maximum compute budget. `diffrax` doesn't necessarily use the maximum budget allowed. 

- **`stochasticity_scale`** : Controls the noise injected at sampling time; `None` (the default) gives the probability-flow **ODE**. Our interpolant is deterministic, so we sample the ODE — there is no score to form an SDE.

We are integrating forward in time to $t = 1$ with the solver.


In [ ]:
MAX_SOLVER_STEPS = 1000

# Rectified flow trains on concrete (z_src, z_tgt) pairs, so we solve enough to fill
# both streams: a DATASET_SIZE training pool and
# a VALIDATION_DATASET_SIZE held-out pool, matching the other couplings
NUM_RECTIFIED_SAMPLES = DATASET_SIZE + VALIDATION_DATASET_SIZE

trained_ema_model = training_loop_POM.ema_model

# Deterministic interpolant ⇒ ODE sampling (stochasticity_scale defaults to
# None): the drift is the bare velocity, with no score term.
solver = Solver(
    SolverConfig(
        stochasticity_scale=None,
        direction=Direction.FORWARD,
        rtol=1e-3,
        atol=1e-3,
        max_steps=MAX_SOLVER_STEPS,
    )
)

key, key_init, key_solve = jr.split(key, 3)

# Two-sided: the source is an explicit distribution (the source ring), so we
# sample it directly for the initial state z_0. The embedders are identity,
# so these raw samples already live in embedding space.
init_keys = jr.split(key_init, NUM_RECTIFIED_SAMPLES)
init_coordinates, init_index = jax.vmap(
    lambda k: sample_coord_and_index_gmm(k, SOURCE_CENTERS)
)(init_keys)
z_init = {
    "coordinates": init_coordinates,
    "index": jax.nn.one_hot(init_index, len(TARGET_CENTERS)),
}


# Run the solver independently on each sample (one PRNG key per sample).
def sample_fn(x, k):
    """Solve a single sample forward with its own PRNG key."""
    return solver(trained_ema_model, x, k)


solve_keys = jr.split(key_solve, NUM_RECTIFIED_SAMPLES)
# `Solver.__call__` decodes back to raw data space internally, so these are
# already raw samples.
raw_samples = jax.vmap(sample_fn)(z_init, solve_keys)

The `"index"` datamode of `raw_samples` is already decoded to label indices, so we one-hot it back onto the convention the (real) raw data uses.

In [ ]:
# The solver decodes discrete modalities straight to label indices; one-hot puts
# them back on the convention the real data uses, so the generated targets are
# drop-in replacements for it.
raw_samples["index"] = jax.nn.one_hot(raw_samples["index"], NUM_MODES)

Let's now turn this into a dataset which `TrainingLoop` can ingest.

In [ ]:
def rectified_pool(
    source: dict[str, jax.Array],
    generated: dict[str, jax.Array],
    sample_slice: slice,
) -> dict[str, tuple[jax.Array, jax.Array]]:
    r"""Pair the ``sample_slice`` of ODE initial conditions with the samples they were mapped to.

    Returns the same ``{modality: (source, target)}`` layout as
    ``sample_gmm_pool``, so ``sample_gmm`` and the grain pipeline below can be
    reused verbatim.
    The difference is what the pairing *means*: source and target are no longer
    independent draws, they are the two endpoints :math:`(z_{\mathrm{src}}^i, z_{\mathrm{tgt}}^i)`
    of one ODE trajectory. That pairing **is** the coupling, and from here
    on it lives in the dataset rather than in a ``Coupling`` object.
    """
    # On device, for the same reason as ``sample_gmm_pool``.
    return {
        modality_name: (
            source[modality_name][sample_slice],
            generated[modality_name][sample_slice],
        )
        for modality_name in source
    }


def gmm_dataset_rectified(
    pool: dict[str, tuple[jax.Array, jax.Array]], size: int, seed: int
):
    """Helper function to instantiate a grain dataset over a fixed pool of rectified pairs.

    Identical to ``gmm_dataset``, except the pool is passed in rather than drawn
    from the GMM. `shuffle` permutes *indices*, and ``sample_gmm`` reads source
    and target at the same index, so the rectified pairing survives shuffling.
    """
    return (
        grain.MapDataset.range(size)
        .seed(seed)
        .shuffle()
        .repeat()
        .batch(batch_size, drop_remainder=True)
        .map(partial(sample_gmm, pool=pool))
        .to_iter_dataset()
    )


# Split the solved pairs: the first DATASET_SIZE pairs train, the rest — exactly
# VALIDATION_DATASET_SIZE of them — are held out. New iterator names, because the
# real-data `train_iter`/`validation_iter` are still needed further below.
train_pool_rectified = rectified_pool(z_init, raw_samples, slice(0, DATASET_SIZE))
validation_pool_rectified = rectified_pool(
    z_init, raw_samples, slice(DATASET_SIZE, None)
)

train_iter_rectified = iter(
    gmm_dataset_rectified(train_pool_rectified, DATASET_SIZE, seed=1)
)
validation_iter_rectified = iter(
    gmm_dataset_rectified(validation_pool_rectified, VALIDATION_DATASET_SIZE, seed=43)
)

In [ ]:
loss_pipeline_rectified = LossPipeline()

gen_model_rectified = copy.deepcopy(gen_model)

logs_history_rectified, io_handler_rectified = make_history_io_handler()

training_loop_rectified = TrainingLoop(
    train_data=train_iter_rectified,
    val_data=validation_iter_rectified,
    loss_pipeline=loss_pipeline_rectified,
    gen_model=gen_model_rectified,
    optimizer_tx=optimizer_tx,
    config=training_loop_cfg,
    io_handler=io_handler_rectified,
)
run_and_time(training_loop_rectified, "Rectified")

## 6. Comparing the couplings

Two costs matter, and a coupling can win on one while losing on the other: what it costs to **train**, and how many solver steps it then needs to **sample**. Whether the additional cost of training is worth the reduced cost in sampling is entirely determined by the constraints of the user.

### 6.1. Cost to train

With all four models trained, we overlay their loss curves. The training objective is the same velocity-MSE in every case — only the coupling (how source/target are paired) differs.

In [ ]:
# Overlay the train/eval loss curves for all four models. Missing keys ->
# nan so the lines break cleanly (the eval-at-start entry has no train_loss; 0
# would render as -inf on the log axis).
histories = {
    "Product-of-marginals": logs_history_POM,
    "MBOT (index)": logs_history_MBOT_index,
    "MBOT (coordinates)": logs_history_MBOT_coordinates,
    "Rectified": logs_history_rectified,
}

# Fixed colour per coupling, reused by every figure from here on.
COUPLING_COLOURS = {
    label: plt.get_cmap("tab10")(index) for index, label in enumerate(histories)
}

fig, (ax_train, ax_eval, ax_cost) = plt.subplots(
    1, 3, figsize=(16, 4), width_ratios=(1, 1, 0.7)
)
ax_train.sharey(ax_eval)
for label, history in histories.items():
    steps = [m["step"] for m in history]
    ax_train.plot(
        steps,
        [m.get("train_loss", float("nan")) for m in history],
        marker="o",
        label=label,
    )
    ax_eval.plot(
        steps,
        [m.get("eval_loss", float("nan")) for m in history],
        marker="o",
        label=label,
    )

for ax, title in [(ax_train, "Train loss"), (ax_eval, "Eval loss")]:
    ax.set_yscale("log")
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.legend(fontsize=8)
ax_train.set_ylabel("Loss (log scale)")

# Third panel: what each coupling cost to train. Rectified flow is stacked,
# because it cannot exist without first training the product-of-marginals model
# — that run is part of its price.
own_cost = [training_seconds[label] for label in histories]
inherited_cost = [
    training_seconds["Product-of-marginals"] if label == "Rectified" else 0.0
    for label in histories
]
positions = range(len(histories))
ax_cost.bar(
    positions,
    own_cost,
    color=[COUPLING_COLOURS[label] for label in histories],
    label="own training",
)
ax_cost.bar(
    positions,
    inherited_cost,
    bottom=own_cost,
    color="none",
    edgecolor="0.35",
    hatch="///",
    linewidth=0.8,
    label="inherited (first model)",
)
for position, (own, inherited) in enumerate(zip(own_cost, inherited_cost, strict=True)):
    ax_cost.text(
        position,
        own + inherited,
        f" {own + inherited:.0f}s",
        ha="center",
        va="bottom",
        fontsize=8,
    )
ax_cost.set_xticks(list(positions))
ax_cost.set_xticklabels([label.replace(" (", "\n(") for label in histories], fontsize=8)
ax_cost.set_ylabel("Training wall clock (s)")
ax_cost.set_title("Cost to train")
ax_cost.margins(y=0.15)
ax_cost.legend(fontsize=8, frameon=False)

fig.suptitle("Coupling comparison: Source Ring -> Target Ring")
plt.tight_layout()
plt.show()

### 6.2. Cost to sample: how many steps does sampling need?

A coupling that pairs nearby points induces **lower-energy transport**: each sample has less distance to cover, so the learned velocity field varies less along the path and a coarse integrator can still follow it. That buys the thing we actually care about — **fewer solver steps for the same sample quality**.

So we measure exactly that. For every trained model we sample with a fixed-step Euler ODE solve ([`ManualSolver`](https://instadeepai.github.io/stix/api_reference/sampling/solver.html)) across a range of step budgets, and score the samples at each budget. `ManualSolver` takes the step count as *configuration*, which is precisely the knob we want to sweep.

Three details keep the comparison fair:

- **one shared source batch** `z_init` for every model, so differences cannot come from the source draw;
- **the same scoring rules** at every budget, with the score of a batch of *true* samples as the reachable reference;
- **ODE sampling throughout** (`stochasticity_scale=None`) — our interpolant is deterministic, so there is no score and no SDE.

What to expect, stated before we look: MBOT (coordinates) pairs *spatially nearest* points, so its transport is shortest and should survive the smallest budget. Rectified flow should be close behind — its dataset-level coupling is an ODE map, and in the limit where that map *is* the straight-line interpolant, a single Euler step is exact. MBOT (index) pairs same-mode points which the $2\pi/3$ rotation places far apart, so semantic alignment should cost it. Product-of-marginals pairs at random and should need the most steps.

> ***The loss curves above are not comparable across couplings.*** MBOT shrinks $\lVert x_1 - x_0 \rVert$ by construction, so it lowers the velocity-MSE whether or not the model is better, and the rectified model is trained on a different dataset entirely. Sampling quality is the only like-for-like verdict.

In [ ]:
# One entry per coupling. Every comparison below iterates over this dict, so
# adding a coupling is a one-line change here.
TRAINED_MODELS = {
    "Product-of-marginals": training_loop_POM.ema_model,
    "MBOT (index)": training_loop_MBOT_index.ema_model,
    "MBOT (coordinates)": training_loop_MBOT_coordinates.ema_model,
    "Rectified": training_loop_rectified.ema_model,
}
NUM_COMPARISON_SAMPLES = 1024
STEP_BUDGETS = (1, 4, 8, 16, 32, 64, 128, 256, 2000)


def sample_source_batch(num_samples: int, key: jax.Array) -> dict[str, jax.Array]:
    r"""Draw a batch of source-ring samples: the solver's initial state z_0."""
    coordinates, index = jax.vmap(
        lambda sample_key: sample_coord_and_index_gmm(sample_key, SOURCE_CENTERS)
    )(jr.split(key, num_samples))
    return {"coordinates": coordinates, "index": jax.nn.one_hot(index, NUM_MODES)}


def fixed_step_ode_solver(num_steps: int) -> ManualSolver:
    """A deterministic Euler solver with a fixed budget of ``num_steps`` steps."""
    return ManualSolver(
        ManualSolverConfig(
            stochasticity_scale=None,
            direction=Direction.FORWARD,
            num_steps=num_steps,
        )
    )


def solve_batch(
    gen_model: GenerativeModel,
    z_init: dict[str, jax.Array],
    key: jax.Array,
    num_steps: int,
) -> dict[str, jax.Array]:
    """Sample a batch in raw data space, one PRNG key per sample."""
    solver = fixed_step_ode_solver(num_steps)
    solve_keys = jr.split(key, z_init["coordinates"].shape[0])
    return jax.vmap(lambda x, solve_key: solver(gen_model, x, solve_key))(
        z_init, solve_keys
    )


def solve_coordinate_paths(
    gen_model: GenerativeModel,
    z_init: dict[str, jax.Array],
    key: jax.Array,
    num_steps: int,
) -> jax.Array:
    """Integrate a batch and return each sample's ``coordinates`` path.

    Shape ``(batch, num_steps + 1, 2)``: ``integrate_with_trajectory`` stores the
    state *before* every step, so the final state is appended to close the path.
    Trajectories live in embedding space — for ``coordinates`` the embedder is the
    identity, so they are already plottable 2D points.
    """
    solver = fixed_step_ode_solver(num_steps)
    solve_keys = jr.split(key, z_init["coordinates"].shape[0])
    z_final, trajectory, _times = jax.vmap(
        lambda x, solve_key: solver.integrate_with_trajectory(gen_model, x, solve_key)
    )(z_init, solve_keys)
    return jnp.concatenate(
        [trajectory["coordinates"], z_final["coordinates"][:, None, :]], axis=1
    )

#### Scoring a batch of samples

Four metrics, each answering a different question. All four are computed from the *generated* `coordinates` and the *generated* mode — the model's own claim about which component it drew from — so they also test whether the two modalities agree with each other.

1. **Log-probability under the claimed mode.** Evaluate the generated coordinate under the Gaussian of the component its own `index` names: $\log \mathcal{N}(x; \mu_{k}, \sigma^2 I)$ with $k$ the predicted mode. A sample scores well only if it lands on a mode *and* labels that mode correctly. Higher is better.
2. **Squared distance to the nearest centre.** $\min_k \lVert x - \mu_k \rVert^2$, averaged. Ignores the label entirely: purely "did the coordinates reach a mode". Lower is better.
3. **Mode-usage KL.** The true index distribution is uniform over the six modes, so we compare the observed histogram against it: $\mathrm{KL}(q \,\Vert\, \mathrm{Uniform}) = \sum_k q_k \log(6 q_k)$. It is $0$ when every mode is used equally and at most $\log 6$ when all mass collapses onto one. Lower is better. (This direction stays finite when a mode is never used, which is exactly the failure we want to plot.)

4. **Index-coordinate agreement.** The fraction of samples whose claimed mode is the one their coordinates actually land nearest: $\frac{1}{N}\sum_i \mathbb{1}[k_i = \arg\min_k \lVert x_i - \mu_k \rVert]$. Metric 1 already punishes disagreement, but mixes it with distance; this isolates it — a model can place every point on a mode and still mislabel one in six. Higher is better.

Metrics 1 and 2 say whether samples are in the right *place*; metric 3 says whether they are spread over modes in the right *proportion*; metric 4 says whether the two modalities agree with each other.

In [ ]:
def log_prob_of_claimed_mode(coordinates: jax.Array, modes: jax.Array) -> float:
    """Mean log density of each coordinate under the mode its own index claims.

    The GMM components are isotropic 2-D Gaussians of width ``GMM_STD``, so the
    log density is available in closed form.
    """
    squared_error = ((coordinates - TARGET_CENTERS[modes]) ** 2).sum(-1)
    log_normaliser = jnp.log(2 * jnp.pi * GMM_STD**2)
    return float(jnp.mean(-squared_error / (2 * GMM_STD**2) - log_normaliser))


def squared_distance_to_nearest_centre(coordinates: jax.Array) -> float:
    """Mean squared distance to the closest target centre, ignoring the index."""
    squared_distances = (
        (coordinates[:, None, :] - TARGET_CENTERS[None, :, :]) ** 2
    ).sum(-1)
    return float(jnp.mean(jnp.min(squared_distances, axis=-1)))


def index_coordinate_agreement(coordinates: jax.Array, modes: jax.Array) -> float:
    """Fraction of samples whose claimed mode is the centre their coordinates are nearest.

    Metric 1 already punishes disagreement between the two modalities, but mixes
    it with distance; this isolates it.
    """
    squared_distances = (
        (coordinates[:, None, :] - TARGET_CENTERS[None, :, :]) ** 2
    ).sum(-1)
    nearest_mode = jnp.argmin(squared_distances, axis=-1)
    return float(jnp.mean(modes == nearest_mode))


def mode_usage_kl(modes: jax.Array) -> float:
    """KL from the observed mode histogram to the true (uniform) one."""
    observed = jnp.bincount(modes, length=NUM_MODES) / modes.size
    return float(
        jnp.sum(jnp.where(observed > 0, observed * jnp.log(observed * NUM_MODES), 0.0))
    )


# One entry per metric: how to score a batch, how to label it, and which way is
# better. The sweep and the figures both read this, so a metric is added once.
METRICS = {
    "log-prob of claimed mode": {
        "score": log_prob_of_claimed_mode,
        "y_scale": "symlog",
        "better": "higher",
    },
    "sq. distance to nearest centre": {
        "score": lambda coordinates, modes: squared_distance_to_nearest_centre(
            coordinates
        ),
        "y_scale": "log",
        "better": "lower",
    },
    "mode-usage KL": {
        "score": lambda coordinates, modes: mode_usage_kl(modes),
        "y_scale": "linear",
        "better": "lower",
    },
    "index-coordinate agreement": {
        "score": index_coordinate_agreement,
        "y_scale": "linear",
        "better": "higher",
    },
}

Now the sweep: every model, every budget, one shared source batch. The solver returns per-mode *probabilities* for the discrete modality, so we take the `argmax` once, explicitly, to get the mode each sample claims.

A batch of true target samples, scored the same way, gives the reachable reference for each metric.

In [ ]:
key, key_source, key_solve = jr.split(key, 3)

# One shared source batch for every model: a paired comparison, so differences
# cannot come from the source draw.
x_init_comparison = sample_source_batch(NUM_COMPARISON_SAMPLES, key_source)

raw_samples_by_coupling = {
    name: {
        num_steps: solve_batch(gen_model, x_init_comparison, key_solve, num_steps)
        for num_steps in STEP_BUDGETS
    }
    for name, gen_model in TRAINED_MODELS.items()
}

# The solver already decodes the discrete modality to the claimed mode.
modes_by_coupling = {
    name: {
        num_steps: raw_samples["index"] for num_steps, raw_samples in by_budget.items()
    }
    for name, by_budget in raw_samples_by_coupling.items()
}

# True samples, scored identically: the reference every metric is measured against.
reference_pool = sample_gmm_pool(NUM_COMPARISON_SAMPLES, seed=7)
reference_coordinates = reference_pool["coordinates"][1]
reference_modes = jnp.argmax(reference_pool["index"][1], axis=-1)

scores = {
    metric_name: {
        name: [
            metric["score"](
                raw_samples_by_coupling[name][num_steps]["coordinates"],
                modes_by_coupling[name][num_steps],
            )
            for num_steps in STEP_BUDGETS
        ]
        for name in TRAINED_MODELS
    }
    for metric_name, metric in METRICS.items()
}
reference_scores = {
    metric_name: metric["score"](reference_coordinates, reference_modes)
    for metric_name, metric in METRICS.items()
}

The metric curves below are the headline result, but they hide *how* each model fails on a tight budget. First, the same samples as scatter plots: one column per coupling, one row per budget, true targets in grey behind.

In [ ]:
SHOWN_BUDGETS = STEP_BUDGETS

# Each generated point is coloured by the mode its own `index` modality claims;
# grey is the true target batch. A well-behaved sample sits in a cluster and claims
# that cluster's mode, so a mis-coloured point is one whose discrete modality
# disagrees with where its coordinates actually landed.
index_cmap = plt.get_cmap("tab10", NUM_MODES)
index_norm = BoundaryNorm([i - 0.5 for i in range(NUM_MODES + 1)], NUM_MODES)

fig, axes = plt.subplots(
    len(SHOWN_BUDGETS),
    len(TRAINED_MODELS),
    figsize=(2.6 * len(TRAINED_MODELS), 2.6 * len(SHOWN_BUDGETS)),
    sharex=True,
    sharey=True,
    layout="constrained",
)
for row, num_steps in enumerate(SHOWN_BUDGETS):
    for col, name in enumerate(TRAINED_MODELS):
        ax = axes[row, col]
        ax.scatter(
            reference_coordinates[:, 0],
            reference_coordinates[:, 1],
            s=6,
            color="0.85",
            linewidths=0,
        )
        generated = raw_samples_by_coupling[name][num_steps]["coordinates"]
        claimed_modes = modes_by_coupling[name][num_steps]
        points = ax.scatter(
            generated[:, 0],
            generated[:, 1],
            s=6,
            c=claimed_modes,
            cmap=index_cmap,
            norm=index_norm,
            alpha=0.8,
            linewidths=0,
        )
        budget_index = STEP_BUDGETS.index(num_steps)
        log_prob = scores["log-prob of claimed mode"][name][budget_index]
        agreement = scores["index-coordinate agreement"][name][budget_index]
        ax.text(
            0.04,
            0.96,
            f"log p = {log_prob:.1f}\nagree = {agreement:.0%}",
            transform=ax.transAxes,
            fontsize=8,
            va="top",
            color="0.25",
            bbox={"facecolor": "white", "alpha": 0.7, "edgecolor": "none", "pad": 1.5},
        )
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(name, fontsize=10)
        if col == 0:
            ax.set_ylabel(
                f"{num_steps} step{'s' if num_steps > 1 else ''}", fontsize=10
            )
fig.colorbar(
    points,
    ax=axes,
    label="claimed mode index",
    ticks=range(NUM_MODES),
    shrink=0.25,
    aspect=12,
)
fig.suptitle(
    "Claimed mode index under a increasing step budget (grey = true targets)",
    fontsize=12,
)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(5 * len(METRICS), 4))
for ax, (metric_name, metric) in zip(axes, METRICS.items(), strict=True):
    for name, curve in scores[metric_name].items():
        ax.plot(
            STEP_BUDGETS,
            curve,
            marker="o",
            markersize=5,
            linewidth=1.8,
            color=COUPLING_COLOURS[name],
            label=name,
        )
    ax.axhline(
        reference_scores[metric_name],
        color="0.4",
        linestyle="--",
        linewidth=1.2,
        label="true samples",
    )
    ax.set_xscale("log", base=2)
    # Per-metric y scale: the log-prob spans orders of magnitude *and* changes
    # sign, the distance is positive and wide-ranging, the KL is bounded.
    ax.set_yscale(metric["y_scale"])
    ax.set_xticks(STEP_BUDGETS)
    ax.set_xticklabels([str(num_steps) for num_steps in STEP_BUDGETS])
    ax.set_xlabel("Euler steps (sampling budget)")
    ax.set_title(f"{metric_name}\n({metric['better']} is better)", fontsize=10)
    ax.grid(True, which="major", alpha=0.25)

# One legend for the whole figure, below the panels, so no curve is covered.
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc="lower center", ncol=len(labels), frameon=False, fontsize=9
)
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

Finally, the paths themselves. A generous budget (`TRAJECTORY_NUM_STEPS`) traces what the learned ODE actually does, for a handful of samples starting from the *same* source points in every panel. The dashed chord is the straight line from start to end: the closer a path hugs it, the less energy the transport spends, and the fewer steps a solver needs to follow it.

In [ ]:
TRAJECTORY_NUM_STEPS = 100
TRAJECTORY_NUM_SAMPLES = 5

key, key_trajectory_source, key_trajectory_solve = jr.split(key, 3)
x_init_trajectory = sample_source_batch(TRAJECTORY_NUM_SAMPLES, key_trajectory_source)

fig, axes = plt.subplots(
    1,
    len(TRAINED_MODELS),
    figsize=(3.2 * len(TRAINED_MODELS), 3.6),
    sharex=True,
    sharey=True,
    layout="constrained",
)
for ax, (name, gen_model) in zip(axes, TRAINED_MODELS.items(), strict=True):
    paths = solve_coordinate_paths(
        gen_model, x_init_trajectory, key_trajectory_solve, TRAJECTORY_NUM_STEPS
    )
    ax.scatter(
        reference_coordinates[:, 0],
        reference_coordinates[:, 1],
        s=5,
        color="0.88",
        linewidths=0,
    )
    for path in paths:
        ax.plot(
            [path[0, 0], path[-1, 0]],
            [path[0, 1], path[-1, 1]],
            color="0.6",
            linestyle="--",
            linewidth=0.8,
        )
        ax.plot(path[:, 0], path[:, 1], color=COUPLING_COLOURS[name], linewidth=1.4)
        ax.scatter(
            path[0, 0],
            path[0, 1],
            s=28,
            facecolors="none",
            edgecolors=COUPLING_COLOURS[name],
            linewidths=1.2,
        )
        ax.scatter(path[-1, 0], path[-1, 1], s=28, color=COUPLING_COLOURS[name])
    ax.set_title(name, fontsize=10)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(
    f"ODE trajectories over {TRAJECTORY_NUM_STEPS} steps "
    "(hollow = source, filled = sample, dashed = straight line)",
    fontsize=11,
)
plt.show()

### 6.3. Discussion of results

Firstly, we caveat all these conclusions with the fact that this is a totally unoptimised notebook with very naive implementations of the `Coupling`s. These methods are introduced purely for illustrative purposes.

Some notes on the performance:

- **MBOT (coordinates) wins overall in terms of sampling performance.** By ~8 steps it is already near the true-sample reference.
- **Product-of-marginals needs the most steps.** Given 256+ steps it tracks the reference as closely as anything else: random pairing costs steps, not quality. Remember: this cost the least to train.
- **MBOT (index) and Rectified plateau *below* the reference.** The $2\pi/3$ rotation makes those pairs long, and the trajectories cross. This coupling wasn't very effective, but does achieve its terminal sampling performance after ~4 steps. MBOT (index) has very good **mode-usage KL**: pairing same-mode points reproduces the mode histogram exactly; it just puts the points in the wrong places.
- **Some reasons Rectified flow might have not worked.** As stated earlier, we have a naive implementation of Rectified flow where we replace one ground-truth dataset with an approximate dataset from the solver. Clearly if the solver performance is not perfect then this creates a hard cap on the performance of the subsequent training. More sophisticated implementations "snap" the samples from the solver to points from the dataset. Further, as we noted previously, due to the non-bijective nature of `from_embeddings_to_raw` (and back via `from_raw_to_embeddings`) we break some of the assumptions of the Rectified flow.
- **The loss curves.** MBOT (coordinates) and Rectified reach the *lowest* velocity-MSE, yet Rectified is among the worst samplers here. Loss is not comparable across couplings: **sampling quality is the only verdict.**
- **What it cost.** MBOT buys its step savings for ~20% more training (33s vs 28s).